In [1]:
import warnings
warnings.simplefilter(action='ignore')
import os, joblib
import numpy as np
import pandas as pd
import polars as pl
import kaggle_evaluation.default_inference_server
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import train_test_split

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

def preprocessing(data, typ, scaler=None, imputer=None):
    
    required_m_cols = [f'M{i}' for i in range(1, 19)]
    required_e_cols = [f'E{i}' for i in range(1, 21)]
    required_i_cols = [f'I{i}' for i in range(1, 10)]
    required_p_cols = ['P8', 'P9', 'P10', 'P12', 'P13']
    required_v_cols = [f'V{i}' for i in range(1, 14)]
    required_s_cols = [f'S{i}' for i in range(1, 13)]
    required_d_cols = [f'D{i}' for i in range(1, 10)]
    
    all_required = required_m_cols + required_e_cols + required_i_cols + required_p_cols + required_v_cols + required_s_cols + required_d_cols
    missing = [col for col in all_required if col not in data.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing[:10]}...")
    
    main_features = (
        [f'E{i}' for i in range(1, 21)] +
        [f'S{i}' for i in range(1, 13)] +
        ['I2', 'P8', 'P9', 'P10', 'P12', 'P13'] +
        ['M*', 'E*', 'I*', 'P*', 'V*', 'S*', 'D*'] +
        ['M*_P*_V*', 'M*_E*_S*', 'M*_P*_I*_V*',
         'V*_M*_E*_I*', 'V*_S*_D*',
         'E*_I*_V*_D*', 'M*_V*_S*_D*',
         'P*_V*_S*', 'P*_M*_D*',
         'M*_E*_P*_S*', 'M*_E*_I*_P*_V*_S*_D*']
    )
    
    data['M*'] = data[[f'M{i}' for i in range(1, 19)]].sum(axis=1)
    data['E*'] = data[[f'E{i}' for i in range(1, 21)]].sum(axis=1)
    data['I*'] = data[[f'I{i}' for i in range(1, 10)]].sum(axis=1)
    data['P*'] = data[['P8', 'P9', 'P10', 'P12', 'P13']].sum(axis=1)
    data['V*'] = data[[f'V{i}' for i in range(1, 14)]].sum(axis=1)
    data['S*'] = data[[f'S{i}' for i in range(1, 13)]].sum(axis=1)
    data['D*'] = data[[f'D{i}' for i in range(1, 10)]].sum(axis=1)
    
    epsilon = 0.01
    data['ME_prod_M*_E*'] = data['M*'] * data['E*']
    data['ME_ratio_M*_E*'] = np.clip(data['M*'] / (data['E*'] + epsilon), -100, 100)
    
    data['MI_prod_M*_I*'] = data['M*'] * data['I*']
    data['MI_ratio_M*_I*'] = np.clip(data['M*'] / (data['I*'] + epsilon), -100, 100)
    data['MI_spread_M*_I*'] = data['M*'] - data['I*']
    
    data['MP_prod_M*_P*'] = data['M*'] * data['P*']
    data['MP_ratio_M*_P*'] = np.clip(data['M*'] / (data['P*'] + epsilon), -100, 100)
    
    data['MV_ratio_M*_V*'] = np.clip(data['M*'] / (data['V*'] + epsilon), -100, 100)
    data['MV_prod_M*_V*'] = data['M*'] * data['V*']
    
    data['MS_prod_M*_S*'] = data['M*'] * data['S*']
    data['MS_weighted_M*_S*'] = data['M*'] * (1 + data['S*'])
    
    data['EI_diff_E*_I*'] = data['E*'] - data['I*']
    data['EI_ratio_E*_I*'] = np.clip(data['E*'] / (data['I*'] + epsilon), -100, 100)
    data['EI_prod_E*_I*'] = data['E*'] * data['I*']
    
    data['EP_prod_E*_P*'] = data['E*'] * data['P*']
    data['EP_ratio_E*_P*'] = np.clip(data['E*'] / (data['P*'] + epsilon), -100, 100)
    
    data['EV_prod_E*_V*'] = data['E*'] * data['V*']
    data['EV_uncertainty_E*_V*'] = np.abs(data['E*']) * data['V*']
    
    data['ES_prod_E*_S*'] = data['E*'] * data['S*']
    data['ES_diff_E*_S*'] = data['E*'] - data['S*']
    
    data['IP_prod_I*_P*'] = data['I*'] * data['P*']
    data['IP_discount_I*_P*'] = data['P*'] / (1 + data['I*'] + epsilon)
    
    data['IV_prod_I*_V*'] = data['I*'] * data['V*']
    data['IS_prod_I*_S*'] = data['I*'] * data['S*']
    
    data['PV_prod_P*_V*'] = data['P*'] * data['V*']
    data['PV_stability_P*_V*'] = np.clip(data['P*'] / (data['V*'] + epsilon), -100, 100)
    
    data['PS_prod_P*_S*'] = data['P*'] * data['S*']
    data['PS_contrarian_P*_S*'] = -data['P*'] * data['S*']
    
    data['VS_prod_V*_S*'] = data['V*'] * data['S*']
    data['VS_panic_V*_S*'] = data['V*'] * np.abs(data['S*'])
    
    interaction_features = [
        'ME_prod_M*_E*', 'ME_ratio_M*_E*', 'MI_prod_M*_I*',
        'MI_ratio_M*_I*', 'MI_spread_M*_I*', 'MP_prod_M*_P*',
        'MP_ratio_M*_P*', 'MV_ratio_M*_V*', 'MV_prod_M*_V*',
        'MS_prod_M*_S*', 'MS_weighted_M*_S*', 'EI_diff_E*_I*',
        'EI_ratio_E*_I*', 'EI_prod_E*_I*', 'EP_prod_E*_P*',
        'EP_ratio_E*_P*', 'EV_prod_E*_V*', 'EV_uncertainty_E*_V*',
        'ES_prod_E*_S*', 'ES_diff_E*_S*', 'IP_prod_I*_P*',
        'IP_discount_I*_P*', 'IV_prod_I*_V*', 'IS_prod_I*_S*',
        'PV_prod_P*_V*', 'PV_stability_P*_V*', 'PS_prod_P*_S*',
        'PS_contrarian_P*_S*', 'VS_prod_V*_S*', 'VS_panic_V*_S*'
    ]
    
    main_features.extend(interaction_features)
    
    data['M*_P*_V*'] = data['M*'] + data['P*'] + data['V*']
    data['M*_E*_S*'] = data['M*'] + data['E*'] + data['S*']
    data['M*_P*_I*_V*'] = data['M*'] + data['P*'] + data['I*'] + data['V*']
    
    data['V*_M*_E*_I*'] = data['V*'] + data['M*'] + data['E*'] + data['I*']
    data['V*_S*_D*'] = data['V*'] + data['S*'] + data['D*']
    
    data['E*_I*_V*_D*'] = data['E*'] + data['I*'] + data['V*'] + data['D*']
    data['M*_V*_S*_D*'] = data['M*'] + data['V*'] + data['S*'] + data['D*']
    
    data['P*_V*_S*'] = data['P*'] + data['V*'] + data['S*']
    data['P*_M*_D*'] = data['P*'] + data['M*'] + data['D*']
    
    data['M*_E*_P*_S*'] = data['M*'] + data['E*'] + data['P*'] + data['S*']
    data['M*_E*_I*_P*_V*_S*_D*'] = data['M*'] + data['E*'] + data['I*'] + data['P*'] + data['V*'] + data['S*'] + data['D*']
    
    if typ == "train":
        data = data[main_features + ["forward_returns"]].copy()
        data = data.rename(columns={'forward_returns': 'target'})
    else:
        data = data[main_features].copy()
    
    if 'target' in data.columns:
        data = data.dropna()
    
    feature_cols = [col for col in data.columns if col != 'target']
    
    if typ == "train":
        imputer = SimpleImputer(strategy='mean')
        data[feature_cols] = imputer.fit_transform(data[feature_cols])
    else:
        if imputer is None:
            raise ValueError("Imputer must be provided for test data")
        data[feature_cols] = imputer.transform(data[feature_cols])
    
    if typ == "train":
        scaler = StandardScaler()
        data[feature_cols] = scaler.fit_transform(data[feature_cols])
    else:
        if scaler is None:
            raise ValueError("Scaler must be provided for test data")
        data[feature_cols] = scaler.transform(data[feature_cols])
    
    if typ == "train":
        return data, scaler, imputer
    else:
        return data


train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')

train_processed, scaler, imputer = preprocessing(train, "train")

from sklearn.model_selection import train_test_split

train_split, val_split = train_test_split(
    train_processed, test_size=0.1, random_state=42
)

train_x = train_split.drop(columns=["target"])
train_y = train_split['target']
test_x = val_split.drop(columns=["target"])
test_y = val_split['target']

In [3]:
train_processed

,E1,E2,E3,E4,E5,E6,E7,E8,E9,E10,...,IP_discount_I*_P*,IV_prod_I*_V*,IS_prod_I*_S*,PV_prod_P*_V*,PV_stability_P*_V*,PS_prod_P*_S*,PS_contrarian_P*_S*,VS_prod_V*_S*,VS_panic_V*_S*,target
6969,-1.549331,0.560517,-0.321032,-0.532785,1.172886,-0.038029,0.557197,-0.611148,-1.168761,-1.416631,...,-0.028003,0.820944,-0.199637,0.070371,-0.003628,-0.783867,0.783867,-0.295836,-0.523958,0.001145
6970,-1.550920,0.566566,-0.314052,-0.542093,1.173788,-0.046697,0.556510,-0.610522,-1.169818,-1.417919,...,-0.021672,0.723391,-0.066899,0.711550,0.042628,-0.290629,0.290629,-0.238670,-0.459183,0.004738
6971,-1.552509,0.613298,-0.262470,-0.551400,1.174689,-0.055365,0.555824,-0.609900,-1.170875,-1.419208,...,-0.019763,0.785069,0.148992,1.007778,0.053831,0.150486,-0.150486,-0.123739,-0.328957,0.006016
6972,-1.554097,0.671845,-0.197419,-0.560708,1.175590,-0.064033,0.557021,-0.589360,-1.171931,-1.420497,...,-0.021941,0.601772,0.258854,0.538325,0.045826,0.124325,-0.124325,-0.099297,-0.301261,0.001414
6973,-1.555685,0.689623,-0.176726,-0.570015,1.176491,-0.072700,0.479651,-0.588772,-1.172988,-1.421786,...,-0.022698,0.571816,0.128379,0.429820,0.039028,-0.104940,0.104940,-0.155070,-0.364458,-0.007182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,-0.191188,0.154333,0.638046,0.211816,-0.593495,-0.055365,0.038864,-0.008678,1.691512,-0.764531,...,-0.016422,-0.076388,-0.256085,0.050413,0.054322,-0.226250,0.226250,-0.251833,-0.474099,0.002457
8986,-0.194929,0.168556,0.651844,0.202509,-0.594396,-0.064033,0.038830,-0.008689,1.692568,-0.765820,...,-0.014489,-0.123606,0.102703,0.057368,0.071541,0.652474,-0.652474,-0.038881,-0.232805,0.002312
8987,-0.198659,0.187698,0.670582,0.193201,-0.595297,-0.072700,0.038785,-0.008699,1.693625,-0.767108,...,-0.020461,-0.204611,-0.035286,-0.334053,0.051598,-0.066999,0.066999,-0.189775,-0.403781,0.002891
8988,-0.202378,0.209330,0.691756,0.183894,-0.596198,-0.081368,0.038540,-0.008710,1.694681,-0.768397,...,-0.011660,-0.172934,-0.447947,0.074871,0.106221,-0.459486,0.459486,-0.407255,-0.650205,0.008310


In [4]:
{'learning_rate': 0.02354764404246205, 
 'n_estimators': 791, 'max_depth': 8,
 'num_leaves': 47, 
 'min_child_samples': 34, 
 'min_data_in_leaf': 57, 
 'subsample': 0.7322242062229307, 
 'colsample_bytree': 0.5776264243251702, 
 'reg_alpha': 2.3297794866835586, 
 'reg_lambda': 0.6547035287542835, 
 'max_bin': 245}

{'learning_rate': 0.02354764404246205,
 'n_estimators': 791,
 'max_depth': 8,
 'num_leaves': 47,
 'min_child_samples': 34,
 'min_data_in_leaf': 57,
 'subsample': 0.7322242062229307,
 'colsample_bytree': 0.5776264243251702,
 'reg_alpha': 2.3297794866835586,
 'reg_lambda': 0.6547035287542835,
 'max_bin': 245}

In [5]:
import numpy as np
import joblib
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score


class ResidualBoostingEnsemble:
    
    def __init__(self, base_params, n_models=3):
        self.base_params = base_params
        self.n_models = n_models
        self.models = []
    
    def fit(self, train_x, train_y, test_x=None, test_y=None):
        current_train_pred = np.zeros(len(train_y))
        
        for i in range(self.n_models):
            print(f'Training Model {i+1}...')
            
            if i == 0:
                target = train_y
            else:
                residuals = train_y - current_train_pred
                target = residuals
            
            model = LGBMRegressor(**self.base_params)
            model.fit(train_x, target)
            self.models.append(model)
            
            pred = model.predict(train_x)
            current_train_pred += pred
            
            train_r2 = mean_absolute_error(train_y, current_train_pred)
            train_mape = mean_absolute_percentage_error(train_y, current_train_pred)
            print(f'Cumulative Training MAE: {train_r2:.4f}')
            print(f'Cumulative Training MAPE: {train_mape:.4f}')
            
            if test_x is not None and test_y is not None:
                current_test_pred = sum(m.predict(test_x) for m in self.models)
                test_r2 = mean_absolute_error(test_y, current_test_pred)
                test_mape = mean_absolute_percentage_error(test_y, current_test_pred)
                print(f'Cumulative Test MAE: {test_r2:.4f}')
                print(f'Cumulative Test MAPE: {test_mape:.4f}')
            
            print()
        
        return self
    
    def predict(self, X):
        predictions = np.zeros(len(X))
        for model in self.models:
            predictions += model.predict(X)
        return predictions
    
    def save(self, filepath):
        joblib.dump(self, filepath)
        print(f'Model saved to {filepath}')
    
    @staticmethod
    def load(filepath):
        return joblib.load(filepath)
    
    def evaluate(self, X, y):
        predictions = self.predict(X)
        r2 = r2_score(y, predictions)
        mape = mean_absolute_percentage_error(y, predictions)
        mae = mean_absolute_error(y, predictions)
        rmse = np.sqrt(mean_squared_error(y, predictions))
        
        print(f'R² Score: {r2:.4f}')
        print(f'MAPE: {mape:.4f}')
        print(f'MAE: {mae:.4f}')
        print(f'RMSE: {rmse:.4f}')
        
        return {'r2': r2, 'mape': mape, 'mae': mae, 'rmse': rmse}

LGBM_R_parm = {
    'boosting_type': 'gbdt', 
    'colsample_bytree': 0.9484106149593443, 
    'learning_rate': 0.1988123373955639, 
    'max_bin': 77, 
    'max_depth': 10, 
    'metric': 'mape', 
    'min_child_samples': 81, 
    'min_data_in_leaf': 21, 
    'n_estimators': 5029, 
    'num_leaves': 42, 
    'objective': 'regression_l1', 
    'reg_alpha': 0.6355835028602363, 
    'reg_lambda': 3.109823217156622, 
    'subsample': 0.7300733288106989, 
    'verbosity': -1
}

ensemble = ResidualBoostingEnsemble(base_params=LGBM_R_parm, n_models=3)
ensemble.fit(train_x, train_y, test_x, test_y)
ensemble.save('LGBM_Residual_Ensemble.pkl')

print('\nFinal Training Evaluation:')
ensemble.evaluate(train_x, train_y)

print('\nFinal Test Evaluation:')
ensemble.evaluate(test_x, test_y)

LGBM_R_model = joblib.load('LGBM_Residual_Ensemble.pkl')

Training Model 1...
Cumulative Training MAE: 0.0017
Cumulative Training MAPE: 29529242.4500
Cumulative Test MAE: 0.0081
Cumulative Test MAPE: 2.2154

Training Model 2...
Cumulative Training MAE: 0.0015
Cumulative Training MAPE: 110737097.3984
Cumulative Test MAE: 0.0080
Cumulative Test MAPE: 2.2229

Training Model 3...
Cumulative Training MAE: 0.0015
Cumulative Training MAPE: 3268975.2101
Cumulative Test MAE: 0.0080
Cumulative Test MAPE: 2.2288

Model saved to LGBM_Residual_Ensemble.pkl

Final Training Evaluation:
R² Score: 0.8563
MAPE: 3268975.2101
MAE: 0.0015
RMSE: 0.0041

Final Test Evaluation:
R² Score: -0.0559
MAPE: 2.2288
MAE: 0.0080
RMSE: 0.0111


In [6]:
SIGNAL_MULTIPLIER = 400.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

def convert_ret_to_signal(ret_value):
    return np.clip(ret_value * SIGNAL_MULTIPLIER + 1, MIN_SIGNAL, MAX_SIGNAL)
    
MIN_INVESTMENT = 0
MAX_INVESTMENT = 2

class ParticipantVisibleError(Exception):
    pass

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str = None) -> float:
    if not pd.api.types.is_numeric_dtype(submission['prediction']):
        raise ParticipantVisibleError('Predictions must be numeric')

    solution = solution.copy()
    solution['position'] = submission['prediction'].values

    if solution['position'].max() > MAX_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].max()} exceeds maximum of {MAX_INVESTMENT}')
    if solution['position'].min() < MIN_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].min()} below minimum of {MIN_INVESTMENT}')

    solution['strategy_returns'] = solution['risk_free_rate'] * (1 - solution['position']) + solution['position'] * solution['forward_returns']

    strategy_excess_returns = solution['strategy_returns'] - solution['risk_free_rate']
    strategy_excess_cumulative = (1 + strategy_excess_returns).prod()
    strategy_mean_excess_return = (strategy_excess_cumulative) ** (1 / len(solution)) - 1
    strategy_std = solution['strategy_returns'].std()

    trading_days_per_yr = 252
    if strategy_std == 0:
        raise ParticipantVisibleError('Division by zero, strategy std is zero')
    sharpe = strategy_mean_excess_return / strategy_std * np.sqrt(trading_days_per_yr)
    strategy_volatility = float(strategy_std * np.sqrt(trading_days_per_yr) * 100)

    market_excess_returns = solution['forward_returns'] - solution['risk_free_rate']
    market_excess_cumulative = (1 + market_excess_returns).prod()
    market_mean_excess_return = (market_excess_cumulative) ** (1 / len(solution)) - 1
    market_std = solution['forward_returns'].std()

    market_volatility = float(market_std * np.sqrt(trading_days_per_yr) * 100)

    if market_volatility == 0:
        raise ParticipantVisibleError('Division by zero, market std is zero')

    excess_vol = max(0, strategy_volatility / market_volatility - 1.2) if market_volatility > 0 else 0
    vol_penalty = 1 + excess_vol

    return_gap = max(
        0,
        (market_mean_excess_return - strategy_mean_excess_return) * 100 * trading_days_per_yr,
    )
    return_penalty = 1 + (return_gap**2) / 100

    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    return min(float(adjusted_sharpe), 1_000_000)


print("\nEvaluating on validation set...")
val_predictions = LGBM_R_model.predict(test_x)
val_signals = np.array([convert_ret_to_signal(pred) for pred in val_predictions])

val_indices = val_split.index
val_original = train.loc[val_indices].copy()

solution_df = pd.DataFrame({
    'forward_returns': val_original['forward_returns'].values,
    'risk_free_rate': val_original['risk_free_rate'].values
})

submission_df = pd.DataFrame({
    'prediction': val_signals
})

try:
    validation_score = score(solution_df, submission_df)
    print(f"Validation Adjusted Sharpe Ratio: {validation_score}")
except Exception as e:
    print(f"Error calculating validation score: {e}")


Evaluating on validation set...
Validation Adjusted Sharpe Ratio: 0.6385291557883038


In [7]:
#normal : 0.7183362427235841

In [8]:
def predict(test: pl.DataFrame) -> float:
    try:
        test_df = test.to_pandas()        
        test_processed = preprocessing(test_df, 'inference', scaler, imputer)
        
        LGBM_R_y_pred = LGBM_R_model.predict(test_processed)
        signal = convert_ret_to_signal(LGBM_R_y_pred)
        
        return float(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        return 1.0 
    
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))